# Phase 3: Statistical Testing & Root Cause Diagnosis

**Objective:**
To determine if the promotional periods (identified via a 15% price drop threshold) actually drove incremental revenue, and whether deeper discounts correlate with higher sales volume. 

**Methodology:**
*   **Independent Samples T-Test:** Comparing average monthly revenue during promotional vs. non-promotional periods for the same products.
*   **Pearson Correlation:** Testing the relationship between discount depth and quantity sold to check for diminishing returns[cite: 1].
*   *Note on Margin:* The standard Online Retail II dataset does not contain Cost of Goods Sold (COGS). Therefore, we proxy margin impact by evaluating overall Revenue Lift vs. Unit Price erosion.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Load the cleaned data
print("Loading data for statistical testing...")
df = pd.read_csv("D:/Retail Pricing & Promotion Analytics/data/processed/cleaned_online_retail.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)

# 2. Recreate the Promo Flag logic in Pandas
# Calculate all-time average price per product
item_avg_price = df.groupby('StockCode')['UnitPrice'].mean().reset_index()
item_avg_price.rename(columns={'UnitPrice': 'AllTimeAvgPrice'}, inplace=True)

# Calculate monthly average price, quantity, and revenue per product
monthly_item_stats = df.groupby(['YearMonth', 'StockCode']).agg(
    MonthlyAvgPrice=('UnitPrice', 'mean'),
    MonthlyQuantity=('Quantity', 'sum'),
    MonthlyRevenue=('Revenue', 'sum')
).reset_index()

# Merge and flag promotional periods (>15% drop from all-time average)
promo_df = pd.merge(monthly_item_stats, item_avg_price, on='StockCode')
promo_df['DiscountDepth'] = (promo_df['AllTimeAvgPrice'] - promo_df['MonthlyAvgPrice']) / promo_df['AllTimeAvgPrice']
promo_df['IsPromo'] = np.where(promo_df['MonthlyAvgPrice'] < (promo_df['AllTimeAvgPrice'] * 0.85), 1, 0)

# 3. Independent Samples T-Test (Promo vs. Non-Promo Revenue)
promo_revenue = promo_df[promo_df['IsPromo'] == 1]['MonthlyRevenue']
non_promo_revenue = promo_df[promo_df['IsPromo'] == 0]['MonthlyRevenue']

# Using equal_var=False (Welch's t-test) as variances between groups are likely different
t_stat, p_val = stats.ttest_ind(promo_revenue, non_promo_revenue, equal_var=False)

print("\n--- T-Test Results: Promotional Lift ---")
print(f"Average Revenue (Non-Promo): £{non_promo_revenue.mean():.2f}")
print(f"Average Revenue (Promo): £{promo_revenue.mean():.2f}")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_val:.4e}")

if p_val < 0.05:
    print("Conclusion: The difference in revenue is STATISTICALLY SIGNIFICANT (p < 0.05).")
else:
    print("Conclusion: The difference in revenue is NOT statistically significant.")

# 4. Pearson Correlation (Discount Depth vs. Quantity Sold)
# Filter for only promotional periods to see if *deeper* discounts move more volume
promo_only = promo_df[promo_df['IsPromo'] == 1]

# Drop NaNs just in case division by zero created anomalies
promo_only_clean = promo_only.dropna(subset=['DiscountDepth', 'MonthlyQuantity'])

corr, p_val_corr = stats.pearsonr(promo_only_clean['DiscountDepth'], promo_only_clean['MonthlyQuantity'])

print("\n--- Pearson Correlation: Discount Depth vs. Volume ---")
print(f"Correlation Coefficient (r): {corr:.4f}")
print(f"P-Value: {p_val_corr:.4e}")

if p_val_corr < 0.05:
    if corr > 0:
        print("Conclusion: Statistically significant POSITIVE correlation. Deeper discounts move more volume.")
    else:
        print("Conclusion: Statistically significant NEGATIVE correlation. Deeper discounts DO NOT move more volume (Diminishing Returns).")
else:
     print("Conclusion: No statistically significant correlation between discount depth and volume.")

Loading data for statistical testing...

--- T-Test Results: Promotional Lift ---
Average Revenue (Non-Promo): £299.22
Average Revenue (Promo): £129.90
T-Statistic: -27.5921
P-Value: 5.3139e-164
Conclusion: The difference in revenue is STATISTICALLY SIGNIFICANT (p < 0.05).

--- Pearson Correlation: Discount Depth vs. Volume ---
Correlation Coefficient (r): -0.0342
P-Value: 3.2883e-02
Conclusion: Statistically significant NEGATIVE correlation. Deeper discounts DO NOT move more volume (Diminishing Returns).
